# Rejected Loan Data 

## Step 1: Setup and Data Loading
We are taking this slowly. In this first step, we will:
1. Import all necessary libraries for the entire pipeline.
2. Configure global settings (random seeds, logging, plotting styles).
3. Define a robust, reusable function to load the dataset using `polars`.

In [ ]:
import os
import logging
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Global Configuration & Setup
# ---------------------------------------------------------

# 1. setup Logging for production-quality tracking
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 2. set Random Seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 3. configure Visualization Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "axes.titlesize": 14,
    "axes.labelsize": 12
})

logging.info("Step 1 Setup complete: Libraries imported and configurations set.")

### 1.1 Modular Data Loading Function
Using `polars` with `infer_schema_length` helps us safely read large CSV files that might have mixed types. We added docstrings and explicit OS path checks for best practices.

In [ ]:
def load_rejected_data(file_path: str) -> pl.DataFrame:
    """
    Loads the rejected loans dataset using Polars for high performance.
    
    Args:
        file_path (str): The absolute or relative path to the CSV file.
        
    Returns:
        pl.DataFrame: The loaded Polars DataFrame.
        
    Raises:
        FileNotFoundError: If the specified file does not exist.
    """
    if not os.path.exists(file_path):
        logging.error(f"File not found at path: {file_path}")
        raise FileNotFoundError(f"Cannot find {file_path}. Please check the path.")
        
    logging.info(f"Starting to load data from {file_path}...")
    
    # infer_schema_length=10000 scans more rows to guess the correct data type
    # null_values handles common missing data strings seamlessly
    df = pl.read_csv(
        file_path, 
        infer_schema_length=10000, 
        null_values=["", "NA", "NaN", "null"], 
        ignore_errors=True
    )
    
    logging.info(f"Successfully loaded data. Shape: {df.shape[0]:,} rows, {df.shape[1]} columns.")
    return df

### 1.2 Execution
Let's execute the function. Make sure your CSV file is in the same directory, or update `rejected_file_path`.

In [ ]:
# Define the file path (Update this if your file is located elsewhere)
rejected_file_path = "../data/raw_data/rejected_2007_to_2018Q4.csv"

# Load the data
try:
    pl_rejected_raw = load_rejected_data(rejected_file_path)
    
    # Display a small preview of the data
    if not pl_rejected_raw.is_empty():
        display(pl_rejected_raw.head())
except Exception as e:
    logging.error(f"Failed to load data: {e}")

## Step 2: Data Cleaning
Now that the data is loaded, we need to clean it. Based on our preview, we will:
1. **Standardize Column Names:** Convert everything to `snake_case`.
2. **Handle Missing Data:** Drop columns that have more than 80% missing values.
3. **Format Dates:** Parse `application_date` into a proper date object.
4. **Clean Numeric Strings:** Strip `%` from `debt_to_income_ratio` and extract years from `employment_length`.
5. **Deduplicate:** Drop exact duplicate rows.

In [ ]:
def standardize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    """Converts all column names to snake_case for consistency."""
    new_names = {col: col.lower().replace(" ", "_").replace("-", "_") for col in df.columns}
    df = df.rename(new_names)
    logging.info("Standardized column names to snake_case.")
    return df

def clean_feature_formats(df: pl.DataFrame) -> pl.DataFrame:
    """Cleans and casts specific columns to their proper data types."""
    transformations = []
    
    # Date Parsing
    if "application_date" in df.columns:
        transformations.append(pl.col("application_date").str.strptime(pl.Date, format="%Y-%m-%d", strict=False))
    
    # DTI Ratio: Remove '%' and cast to float
    if "debt_to_income_ratio" in df.columns:
        transformations.append(
            pl.col("debt_to_income_ratio").str.replace("%", "", literal=True).cast(pl.Float64, strict=False)
        )
        
    # Employment Length: Extract numbers
    if "employment_length" in df.columns:
        transformations.append(
            pl.col("employment_length")
            .str.replace("< 1", "0", literal=True)
            .str.extract(r"(\d+)")
            .cast(pl.Int8, strict=False)
            .alias("employment_years") # Rename to a more descriptive name
        )
        
    # Zip Code: Extract 3-digit prefix from masked zips (e.g., '481xx' -> '481')
    if "zip_code" in df.columns:
        transformations.append(pl.col("zip_code").str.slice(0, 3).alias("zip_code_prefix"))
        
    # Loan Title: Basic text cleaning (lowercase)
    if "loan_title" in df.columns:
        transformations.append(pl.col("loan_title").str.to_lowercase())
    
    if transformations:
        df = df.with_columns(transformations)
        logging.info("Applied format cleaning to date, DTI, employment, zip, and title columns.")
        
    return df

### 2.2 Execute Cleaning Pipeline
Now we orchestrate the cleaning functions and convert the result to a Pandas DataFrame for the next steps.

In [ ]:
if 'pl_rejected_raw' in locals() and not pl_rejected_raw.is_empty():
    logging.info("Executing data cleaning pipeline...")
    
    # --- Cleaning Pipeline (Simplified to prevent memory crashes) ---
    pl_rejected_clean = (
        pl_rejected_raw
        .pipe(standardize_column_names)
        .pipe(clean_feature_formats)
    )
    
    # Drop original columns that were transformed
    pl_rejected_clean = pl_rejected_clean.drop(['employment_length', 'zip_code'])
    
    logging.warning("Skipping .unique() to prevent memory issues. The dataset may contain duplicates.")
    
    # --- Create a Sample for EDA to prevent kernel crash ---
    SAMPLE_SIZE = 200_000 # Using a 200k sample for EDA
    logging.info(f"Creating a random sample of {SAMPLE_SIZE:,} rows for EDA to conserve memory...")
    
    if pl_rejected_clean.height > SAMPLE_SIZE:
        df_rejected_sample = pl_rejected_clean.sample(n=SAMPLE_SIZE, seed=RANDOM_SEED).to_pandas()
    else:
        df_rejected_sample = pl_rejected_clean.to_pandas()
    
    logging.info(f"Sampled Pandas DataFrame created with shape: {df_rejected_sample.shape}")
    
    # --- Display Cleaned Sample Data ---
    display(df_rejected_sample.head())
    display(df_rejected_sample.info())
else:
    logging.warning("Raw data is not available. Please run Step 1 successfully first.")
    df_rejected_sample = pd.DataFrame()

### 2.3 Verify Data Cleaning
Let's run a quick check to confirm the remaining missing values percentage and duplicate counts in our sample.

In [ ]:
if 'df_rejected_sample' in locals() and not df_rejected_sample.empty:
    print("--- Data Cleaning Verification ---")
    print(f"Total Rows in Sample: {len(df_rejected_sample):,}")
    print(f"Duplicate Rows: {df_rejected_sample.duplicated().sum():,}")
    
    missing_pct = (df_rejected_sample.isnull().sum() / len(df_rejected_sample)) * 100
    print("\nMissing Values (%):")
    print(missing_pct[missing_pct > 0].sort_values(ascending=False).round(2))
    print("----------------------------------")
else:
    logging.warning("Sample DataFrame not found. Run the cleaning step first.")

### 2.4 Handling Missing Values for EDA
With nearly **67% missing data in `risk_score`**, we cannot simply drop rows with missing values, as we would lose most of our dataset. 
- **For EDA:** We will leave `risk_score` as `NaN` (visualization libraries like Seaborn handle NaNs gracefully).
- **For ML (Later):** Our Scikit-Learn pipeline will handle imputing this feature.
- **For `employment_years`:** Since only ~3.5% is missing, we can safely impute it with the median value right now to clean up the sample.

In [ ]:
if 'df_rejected_sample' in locals() and not df_rejected_sample.empty:
    logging.info("Imputing missing values for employment_years...")
    
    # Fill missing employment_years with the median
    emp_median = df_rejected_sample['employment_years'].median()
    df_rejected_sample['employment_years'] = df_rejected_sample['employment_years'].fillna(emp_median)
    
    print(f"Missing values in 'employment_years' after imputation: {df_rejected_sample['employment_years'].isnull().sum()}")
else:
    logging.warning("Sample DataFrame not found.")

## Save files

In [ ]:
# if 'df_rejected_sample' in locals() and not df_rejected_sample.empty:
#     # Convert pandas DataFrame to Polars DataFrame
#     pl_rejected_sample = pl.from_pandas(df_rejected_sample)
#     logging.info(f"Converted pandas DataFrame to Polars DataFrame. Shape: {pl_rejected_sample.shape}")
#     print(f"Polars DataFrame created with shape: {pl_rejected_sample.shape}")
#     display(pl_rejected_sample.head())
# else:
#     logging.warning("Sample DataFrame not found. Run the cleaning step first.")

In [ ]:
# if 'pl_rejected_sample' in locals():
#     pl_rejected_sample.write_parquet("../data/processed_data/processed_rejected_df.parquet")
#     logging.info("Saved Polars DataFrame to parquet file.")

In [ ]:
if 'df_rejected_sample' in locals():
    df_rejected_sample.to_csv("../data/processed_data/processed_rejected_df.csv")
    logging.info("Saved CSV DataFrame to parquet file.")

In [ ]:
df_rejected_sample.dtypes